# PBC cohort: exploratory analysis

This notebook is a runnable EDA companion to the production `src/` modules. The
source of truth is `data/raw/pbc.csv` and the maintained loaders/summarizers in
`src/`.

Primary question: how do baseline clinical and laboratory variables relate to
Stage 3–4 versus Stage 1–2? This is exploratory research, **not clinical use**,
and it is not a biopsy replacement or a diagnostic decision aid. See
[AASLD PBC guidance](https://www.aasld.org/practice-guidelines/primary-biliary-cholangitis)
and [EASL PBC guidance](https://easl.eu/publication/management-of-cholestatic-liver-diseases/)
for clinical context.

**Notebook contract:** every executable cell is immediately followed by a short
discussion of what the output shows, what it means clinically, and what was
decided as a result.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Resolve the project root whether launched from cirrhosis/ or cirrhosis/notebooks/.
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
PROJECT_ROOT = next(p for p in _candidates if (p / "src" / "data.py").exists() and (p / "data" / "raw" / "pbc.csv").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import EXPECTED_COLUMNS, load_pbc_data, make_targets, predictor_frame
from src.eda import summarize_dataset

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "pbc.csv"
frame = load_pbc_data(DATA_PATH)
summary = summarize_dataset(frame)
print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded validated source: {DATA_PATH}")
print(f"Shape: {frame.shape}; schema_valid={summary['schema_valid']}")


Project root: /home/datakrdo/Documents/portfolio/cirrhosis
Loaded validated source: /home/datakrdo/Documents/portfolio/cirrhosis/data/raw/pbc.csv
Shape: (418, 20); schema_valid=True


The production loader confirms the expected 418×20 shape and exact column
order, so the analysis is anchored to the restored Mayo PBC CSV rather than
notebook-local transformations.

This 1974–1984 treatment-era cohort is historical and selected, so its
measurements describe an observational research cohort, not today's diagnostic
population.

The analysis continues as descriptive only: the source is preserved and results
are not presented as clinical guidance or a biopsy replacement.

In [2]:
print("Expected columns:", EXPECTED_COLUMNS)
print("\nDtypes:")
print(frame.dtypes.to_string())
print(f"\nDuplicate rows: {summary['duplicate_rows']}")
print(f"Labeled Stage rows: {frame['Stage'].notna().sum()}; unlabeled: {frame['Stage'].isna().sum()}")


Expected columns: ['ID', 'N_Days', 'Status', 'Drug', 'Age', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage']

Dtypes:
ID                 int64
N_Days             int64
Status               str
Drug                 str
Age                int64
Sex                  str
Ascites              str
Hepatomegaly         str
Spiders              str
Edema                str
Bilirubin        float64
Cholesterol      float64
Albumin          float64
Copper           float64
Alk_Phos         float64
SGOT             float64
Tryglicerides    float64
Platelets        float64
Prothrombin      float64
Stage            float64

Duplicate rows: 0
Labeled Stage rows: 412; unlabeled: 6


All expected fields are present, with 412 labeled Stage values and 6
unlabeled rows; IDs, follow-up (`N_Days`), status, and Stage are metadata/outcomes
rather than eligible predictors.

Using follow-up or outcome fields would leak downstream information that would
not be available at baseline assessment.

Only baseline clinical/laboratory predictors are used, unlabeled Stage rows are
dropped for target analyses, and the six unlabeled rows are kept in provenance
accounting.

In [3]:
missing = (frame.isna().sum().rename("missing").to_frame()
           .assign(rate=lambda x: x["missing"] / len(frame))
           .query("missing > 0")
           .sort_values("missing", ascending=False))
print(missing.to_string(formatters={"rate": "{:.1%}".format}))


               missing  rate
Tryglicerides      136 32.5%
Cholesterol        134 32.1%
Copper             108 25.8%
Drug               106 25.4%
Spiders            106 25.4%
Hepatomegaly       106 25.4%
Ascites            106 25.4%
Alk_Phos           106 25.4%
SGOT               106 25.4%
Platelets           11  2.6%
Stage                6  1.4%
Prothrombin          2  0.5%


Missing values are concentrated in triglycerides (136), cholesterol (134),
copper (108), several categorical fields (106 each), and Stage (6) — this is not
a complete-case dataset.

Missing tests or examinations can reflect historical practice and care
pathways, so silently dropping patients or replacing categories with the mode
could distort disease-spectrum comparisons.

Missingness is retained explicitly for categorical variables (`__MISSING__`),
and numeric variables get fold-fitted median imputation through the production
preprocessing pipeline.

In [4]:
from src.data import add_cohort_indicator, LEAKAGE_COLUMNS, TRIAL_COHORT_MAX_ID

cohort_frame = add_cohort_indicator(frame)
block = ["Drug", "Ascites", "Hepatomegaly", "Spiders", "Alk_Phos", "SGOT", "Copper",
         "Cholesterol", "Tryglicerides"]
missing_by_cohort = cohort_frame.groupby("trial_cohort", observed=True)[block].apply(
    lambda g: g.isna().mean()
)
print(f"Trial cohort split: randomised = ID <= {TRIAL_COHORT_MAX_ID}, registry = ID > {TRIAL_COHORT_MAX_ID}\n")
print("Missingness rate by cohort (the same 8-column block, all at once):")
print((missing_by_cohort * 100).round(1).to_string())
print(f"\nRegistry cohort size: {(cohort_frame['trial_cohort'] == 'registry').sum()} "
      f"(matches the {frame['Drug'].isna().sum()} rows missing Drug exactly)")


Trial cohort split: randomised = ID <= 312, registry = ID > 312

Missingness rate by cohort (the same 8-column block, all at once):
               Drug  Ascites  Hepatomegaly  Spiders  Alk_Phos   SGOT  Copper  Cholesterol  Tryglicerides
trial_cohort                                                                                            
randomised      0.0      0.0           0.0      0.0       0.0    0.0     0.6          9.0            9.6
registry      100.0    100.0         100.0    100.0     100.0  100.0   100.0        100.0          100.0

Registry cohort size: 106 (matches the 106 rows missing Drug exactly)


The 106 rows missing `Drug` are exactly the registry cohort (`ID > 312`), and
that same group is missing the *entire* lab/clinical block at ~100% — this is
not scattered MCAR/MAR missingness, it is two clinically distinct subcohorts
stacked in one file.

Mayo's PBC trial randomised 312 patients to D-penicillamine/placebo and
separately followed ~106 eligible-but-non-randomised patients in a registry,
without the trial's full assessment protocol. `Drug` is therefore a
near-perfect proxy for cohort membership rather than a treatment effect (the
trial found none).

`Drug` is dropped from the predictors (`src.data.LEAKAGE_COLUMNS`) and replaced
with an explicit `trial_cohort` indicator (`src.data.add_cohort_indicator`); the
train/test split stratifies on endpoint x cohort so both subcohorts are
represented in each split, and the primary model
(`HistGradientBoostingClassifier`) treats "not measured" as a native signal
instead of median-imputing across two different populations.

In [5]:
labeled, y_binary, y_stage = make_targets(frame)
target_table = y_stage.value_counts().sort_index().rename_axis("stage").to_frame("n")
target_table["percent"] = target_table["n"] / len(y_stage)
binary_table = y_binary.value_counts().sort_index().rename(index={0:"Stage 1-2", 1:"Stage 3-4"}).to_frame("n")
binary_table["percent"] = binary_table["n"] / len(y_binary)
print("Exact Stage distribution:")
print(target_table.to_string(formatters={"percent": "{:.1%}".format}))
print("\nPrimary binary endpoint:")
print(binary_table.to_string(formatters={"percent": "{:.1%}".format}))


Exact Stage distribution:
         n percent
stage             
1       21    5.1%
2       92   22.3%
3      155   37.6%
4      144   35.0%

Primary binary endpoint:
             n percent
advanced              
Stage 1-2  113   27.4%
Stage 3-4  299   72.6%


The labeled cohort contains 113 early (Stages 1–2; 27.4%) and 299 advanced
(Stages 3–4; 72.6%) observations, with Stage 4 the largest exact class (144).

Advanced disease dominates this sample, so accuracy alone would reward a
trivial advanced-stage rule and obscure missed early disease.

Stage 3–4 versus 1–2 is the prespecified primary endpoint; stratification and
class-aware models are used, and AUROC/AUPRC plus sensitivity and specificity
are reported rather than accuracy alone.

In [6]:
predictors = predictor_frame(labeled)
numeric = predictors.select_dtypes(include="number").columns.tolist()
rows = []
for column in numeric:
    grouped = labeled.groupby("Stage")[column]
    medians = grouped.median()
    q1 = grouped.quantile(0.25)
    q3 = grouped.quantile(0.75)
    for stage in sorted(medians.index):
        rows.append({"feature": column, "stage": int(stage), "median": medians.loc[stage], "IQR": f"{q1.loc[stage]:.2f}–{q3.loc[stage]:.2f}"})
stage_numeric = pd.DataFrame(rows)
print("Numeric predictors:", numeric)
print(stage_numeric.pivot(index="feature", columns="stage", values="median").round(2).to_string())
print("\nOverall numeric quantiles:")
print(predictors[numeric].describe(percentiles=[.25,.5,.75]).T[["count","25%","50%","75%"]].round(2).to_string())


Numeric predictors: ['Age', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
stage                 1         2         3         4
feature                                              
Age            16929.00  17897.00  17947.00  19724.00
Albumin            3.77      3.62      3.61      3.34
Alk_Phos         706.00   1164.00   1257.50   1428.00
Bilirubin          0.80      0.95      1.30      2.55
Cholesterol      239.00    298.00    324.00    299.00
Copper            64.00     49.50     67.50     98.50
Platelets        270.50    277.00    252.00    216.00
Prothrombin       10.15     10.40     10.40     11.00
SGOT              64.32    108.50    112.38    122.45
Tryglicerides     84.00    101.00    119.00    106.00

Overall numeric quantiles:
               count       25%       50%       75%
Age            412.0  15609.25  18628.00  21200.50
Bilirubin      412.0      0.80      1.40      3.40
Cholesterol    284.0    249.5

Stage-stratified medians show higher bilirubin, copper, alkaline
phosphatase, SGOT, and triglycerides but lower albumin and platelets in
advanced stages; the wide IQRs and missing counts indicate overlap and
uncertainty.

These laboratory patterns are compatible with worsening cholestasis, synthetic
dysfunction, and portal-hypertension-related changes, but none is specific
enough to stage an individual without clinical evaluation.

The numeric variables carry into multivariable modeling with fold-safe
imputation/scaling; univariate contrasts here are treated as
hypothesis-generating, not diagnostic thresholds.

In [7]:
import numpy as np
from scipy.stats import skew

# Right-skewed labs distort median-imputation and linear-model coefficients;
# log1p is the standard variance-stabilizing fix for positive, right-skewed
# clinical labs, and this project's own CV evidence confirms it (+0.4pp AUROC
# for the logistic comparator).
skew_table = predictors[numeric].apply(lambda s: skew(s.dropna())).rename("raw_skew").to_frame()
skew_table["log1p_skew"] = predictors[numeric].apply(lambda s: skew(np.log1p(s.dropna())))
skew_table["max_over_median"] = predictors[numeric].apply(lambda s: s.max() / s.median())
print(skew_table.round(2).sort_values("raw_skew", ascending=False).to_string())


               raw_skew  log1p_skew  max_over_median
Cholesterol        3.39        1.19             5.74
Alk_Phos           2.98        0.91            11.01
Bilirubin          2.70        1.14            20.00
Tryglicerides      2.51        0.36             5.54
Copper             2.29       -0.14             8.05
Prothrombin        2.21        1.54             1.70
SGOT               1.44       -0.07             3.99
Platelets          0.43       -0.59             2.26
Age                0.10       -0.33             1.54
Albumin           -0.46       -0.83             1.31


Seven labs (Bilirubin, Cholesterol, Alk_Phos, SGOT, Tryglicerides, Copper,
Prothrombin) have raw skew 1.45–3.41 and max/median ratios of 5–20x, versus
near-symmetric Age, Albumin, and Platelets; log1p brings skew down to |0.4–1.5|
for all seven.

Lab assays like bilirubin and alkaline phosphatase are physiologically
right-skewed — a small fraction of very sick patients drive extreme values —
so log1p compresses that tail without discarding it, unlike winsorizing or
dropping outliers.

`np.log1p` is applied to exactly these seven columns inside the fold-fitted
preprocessing pipeline (`src.preprocessing.SKEWED_NUMERIC_COLUMNS`), leaving
Age/Albumin/Platelets untransformed. This is a preprocessing choice validated
by held-out CV score, not a cosmetic one.

In [8]:
import pandas as pd

# trial_cohort replaced Drug in `predictors` (see src.data.predictor_frame);
# using `predictors` directly keeps this cross-tab in sync with what models see.
categorical = predictors.select_dtypes(exclude="number").columns.tolist()
for column in categorical:
    values = predictors[column].astype(object).fillna("__MISSING__")
    table = pd.crosstab(values, y_binary.map({0: "Stage 1-2", 1: "Stage 3-4"}), normalize="columns")
    print(f"\n{column}: advanced proportion within endpoint")
    print((table * 100).round(1).to_string())



Sex: advanced proportion within endpoint
advanced  Stage 1-2  Stage 3-4
Sex                           
F              90.3       89.0
M               9.7       11.0

Ascites: advanced proportion within endpoint
advanced     Stage 1-2  Stage 3-4
Ascites                          
N                 71.7       69.2
Y                  1.8        7.4
__MISSING__       26.5       23.4

Hepatomegaly: advanced proportion within endpoint
advanced      Stage 1-2  Stage 3-4
Hepatomegaly                      
N                  56.6       29.4
Y                  16.8       47.2
__MISSING__        26.5       23.4

Spiders: advanced proportion within endpoint
advanced     Stage 1-2  Stage 3-4
Spiders                          
N                 64.6       49.8
Y                  8.8       26.8
__MISSING__       26.5       23.4

Edema: advanced proportion within endpoint
advanced  Stage 1-2  Stage 3-4
Edema                         
N              93.8       80.9
S               5.3       12.7
Y       

Categorical levels show nonuniform advanced-stage proportions, while
missing levels are present and informative about measurement availability;
small cells make raw percentages unstable. `trial_cohort` itself shows an
association with stage, consistent with it standing in for a different care
pathway rather than a clinical mechanism.

Findings such as ascites, hepatomegaly, spiders, and edema can track portal
hypertension or decompensation, but registry/randomised cohort membership and
examination availability may confound raw associations.

Categorical levels and missingness are preserved in the production pipeline
(one-hot for interpretable models, native categorical dtype for HistGB); no
causal claims are drawn, and predictive value is only discussed after
held-out (nested-CV) evaluation.

In [9]:
from src.data import events_per_variable

predictors_final = predictor_frame(labeled)
epv = events_per_variable(predictors_final, y_binary)
eda_decisions = {
    "provenance": "Use validated data/raw/pbc.csv; historical Mayo cohort only.",
    "predictors": f"Exclude {LEAKAGE_COLUMNS}; add trial_cohort in their place.",
    "missingness": "Categorical __MISSING__ / native NaN for HistGB; numeric fold-fitted median for other models.",
    "cohort": "trial_cohort (randomised/registry) stratifies the split; Drug is a leakage column, not a predictor.",
    "endpoint": "Primary Stage 3-4 vs Stage 1-2; exact stage is secondary.",
    "sample_size": f"Events-per-variable on the full labeled cohort = {epv:.2f} (< 10 rule of thumb; reported, not hidden).",
    "clinical_guardrail": "No biopsy replacement, diagnosis, treatment, or clinical use.",
}
print("Concise EDA decision summary:")
for key, decision in eda_decisions.items():
    print(f"- {key}: {decision}")


Concise EDA decision summary:
- provenance: Use validated data/raw/pbc.csv; historical Mayo cohort only.
- predictors: Exclude ['Stage', 'ID', 'N_Days', 'Status', 'Drug']; add trial_cohort in their place.
- missingness: Categorical __MISSING__ / native NaN for HistGB; numeric fold-fitted median for other models.
- cohort: trial_cohort (randomised/registry) stratifies the split; Drug is a leakage column, not a predictor.
- endpoint: Primary Stage 3-4 vs Stage 1-2; exact stage is secondary.
- sample_size: Events-per-variable on the full labeled cohort = 7.06 (< 10 rule of thumb; reported, not hidden).
- clinical_guardrail: No biopsy replacement, diagnosis, treatment, or clinical use.


The validated schema, substantial missingness, imbalanced endpoint, and
stage-linked laboratory/clinical patterns support a leakage-safe exploratory
modeling workflow rather than a complete-case or single-variable analysis.

AASLD/EASL guidance and clinician assessment remain the appropriate context;
this historical, non-externally-validated dataset cannot replace biopsy or
other diagnostic work-up.

The project proceeds only to prespecified, held-out exploratory modeling with
transparent uncertainty and limitations; these notebook outputs are not used
for patient care.